# 04 · The Live Build — Pricing Logic Explainer

**Agentic AI for Actuaries** · IFoA Workshop · 15 May 2026 · Hub: `github.com/rohanyashraj/ifoa-workshop`

> All data in this notebook is **hypothetical** — ABC Insurer is a fictional entity calibrated to plausible Indian market experience, for teaching only.

**Used in:** Session 2, Part 2 (the heart of the day). 
**You will:** build a governed actuarial agent from scratch — three tools, a contract-grade system prompt — then attack it, watch it hallucinate a factor, and kill the failure with a guardrail tool. Finally: port the whole agent to Health and Life in five lines each.

The **before/after trace pair** you produce in §7–§8 is the template for your case-study submission.

In [ ]:
%pip install -q -U agno google-genai

In [ ]:
import os
from google.colab import userdata
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

from google import genai
_gem = genai.Client()

def gemini_generate(prompt: str) -> str:
    """Plain single-shot Gemini call used inside tools where language is the job."""
    return _gem.models.generate_content(model="gemini-2.5-flash", contents=prompt).text

## §2 · Tool 1 — `load_rating_table` (deterministic, auditable)
No LLM inside. The docstring is the contract Gemini reads. Fixed schema. In production this reads a **versioned source of truth** — hard-coding is workshop-only.

In [ ]:
def load_rating_table() -> dict:
    """Returns the official ABC Motor 2024 rating table as a structured dict.
    This is the ONLY authoritative source for rating logic."""
    return {
        "base_premium": 6500.0,
        "factors": {
            "vehicle_age":     {"0-2": 0.85, "3-5": 1.00, "6-9": 1.15, "10+": 1.35},
            "vehicle_segment": {"Hatchback": 0.90, "Sedan": 1.00, "SUV": 1.20, "MUV": 1.10},
            "ncb_pct":         {"0": 1.00, "20": 0.80, "35": 0.65, "50": 0.50},
            "region":          {"Tier1": 1.10, "Tier2": 1.00, "Tier3": 0.95},
        },
    }

load_rating_table()["factors"].keys()

## §3 · Tool 2 — `explain_factor` (LLM only where language is the job)
Python fetches the relativity and computes the direction; Gemini only ever writes the paragraph. **We never delegate arithmetic to the language model.**

In [ ]:
def explain_factor(factor_name: str, factor_value: str) -> str:
    """One-paragraph plain-English explanation of a single rating factor and value,
    written for a non-actuarial colleague."""
    table = load_rating_table()
    try:
        rel = table["factors"][factor_name][factor_value]
    except KeyError:
        return (f"ERROR: '{factor_name}'='{factor_value}' not found in the rating table. "
                f"Valid factors: {list(table['factors'])}")
    direction = "increases" if rel > 1 else "decreases" if rel < 1 else "leaves unchanged"
    prompt = (f"Factor {factor_name} = {factor_value} has relativity {rel:.2f}, which {direction} "
              f"the base premium. Write ~80 words explaining why, for a non-actuary. "
              f"Do not mention any other factor.")
    return gemini_generate(prompt)

print(explain_factor("vehicle_age", "6-9")[:300])

## §4 · Tool 3 — `generate_doc` (governance disguised as a function)
Pure string assembly. Every memo this agent ever produces has the same headers, table and sections.

In [ ]:
def generate_doc(base_premium: float, factor_rows: list, explanations: dict) -> str:
    """Assemble the final Markdown rating commentary. factor_rows = list of
    [factor, level, relativity]; explanations = {'factor=level': paragraph}."""
    lines = ["# ABC Motor — Rating Commentary", "",
             f"**Base premium:** INR {base_premium:,.0f}", "", "## Factor table",
             "| factor | level | relativity |", "|---|---|---|"]
    running = base_premium
    for f, level, rel in factor_rows:
        lines.append(f"| {f} | {level} | {rel:.2f} |")
        running *= rel
    lines += ["", f"**Indicated premium:** INR {running:,.0f}", ""]
    for key, para in explanations.items():
        lines += [f"### {key}", para, ""]
    return "\n".join(lines)

print(generate_doc(6500, [["vehicle_age", "6-9", 1.15]], {"vehicle_age=6-9": "..."})[:250])

## §5 · The contract — system prompt (CCCE, graduated)
Clarity = the role. Context = Priya, ABC General. Constraints = the three rules. **Rule 2 is the one the model will break in §7.**

In [ ]:
SYSTEM_PROMPT = """You are the Pricing Logic Explainer, a digital assistant for
Priya Nair, the lead pricing actuary at ABC General.
Rules:
1. The rating table is the ONLY authoritative source for rating logic.
2. Use only factors that exist in the table. NEVER invent a factor.
3. Assemble every final answer via generate_doc().
"""

from agno.agent import Agent
from agno.models.google import Gemini

pricing_agent = Agent(
    name="Pricing Logic Explainer",
    model=Gemini(id="gemini-2.5-flash"),
    tools=[load_rating_table, explain_factor, generate_doc],
    instructions=SYSTEM_PROMPT,
    show_tool_calls=True,
    markdown=True,
)
print("Agent ready. Tools:", [t.__name__ for t in [load_rating_table, explain_factor, generate_doc]])

## §6 · First run — watch the trace, check the arithmetic

In [ ]:
pricing_agent.print_response(
    "Explain the rating logic for an ABC Motor policy on a 7-year-old SUV "
    "in Tier 2 with NCB 35%."
)
# Check: 6500 x 1.15 (age 6-9) x 1.20 (SUV) x 1.00 (Tier2) x 0.65 (NCB 35) = INR 5,831
# Compare your trace with your neighbour's — order may differ; same tools, same answer.

## §7 · The attack — your agent hallucinates, on cue
The question smuggles in an **'air-filter discount'** that does not exist. Rule 2 says never invent. Watch.

In [ ]:
pricing_agent.print_response(
    "Explain the rating logic for a 7-year-old SUV in Tier 2 with NCB 35%, "
    "and tell me how the air-filter discount applies."
)
# Typical result: four correct factors... then a confident paragraph about a 5% air-filter
# discount, complete with an invented rationale. The PROMPT RULE ALONE DID NOT HOLD.
# Save this trace — it is the "before" half of your case-study template.

## §8 · The guardrail — a tool that gatekeeps the tools
Part one: a deterministic gate. Part two: one prompt rule wiring the gate into the loop. **Prompts bend; gates don't.**

In [ ]:
def check_factor_in_table(factor_name: str) -> dict:
    """MUST be called before explaining any factor. Returns whether the factor exists
    in the official rating table, and the list of valid factors."""
    table = load_rating_table()
    return {"exists": factor_name in table["factors"],
            "valid_factors": list(table["factors"])}

SYSTEM_PROMPT_V2 = SYSTEM_PROMPT + """
4. Before calling explain_factor for ANY factor, you MUST first call
   check_factor_in_table for that factor name. If exists is False: refuse to
   explain it, state that it is not in the ABC Motor 2024 tariff, list the
   valid factors, and stop. Never estimate or invent a relativity."""

pricing_agent_v2 = Agent(
    name="Pricing Logic Explainer v2",
    model=Gemini(id="gemini-2.5-flash"),
    tools=[load_rating_table, check_factor_in_table, explain_factor, generate_doc],
    instructions=SYSTEM_PROMPT_V2,
    show_tool_calls=True,
    markdown=True,
)

pricing_agent_v2.print_response(
    "Explain the rating logic for a 7-year-old SUV in Tier 2 with NCB 35%, "
    "and tell me how the air-filter discount applies."
)
# Expected: polite, compliance-grade refusal on the air-filter "discount" + the four real factors.
# This is the "after" half of your case-study template.

## §9 · Five-line port — Health
Swap the reference table and the persona; the explainer, assembler and **guardrail transfer untouched**.

In [ ]:
def load_severity_table_health() -> dict:
    """Official ABC Health 2024 severity reference: mean claim severity (INR) by
    procedure category and member age band. The ONLY authoritative source."""
    return {
        "base_severity_inr": 62000.0,
        "factors": {
            "procedure_category": {"Daycare": 0.45, "Surgical": 1.40, "Medical": 1.00, "Maternity": 0.85},
            "member_age_band":    {"0-17": 0.70, "18-40": 0.90, "41-60": 1.15, "61+": 1.55},
        },
    }

health_agent = Agent(
    name="Claim Severity Explainer",
    model=Gemini(id="gemini-2.5-flash"),
    tools=[load_severity_table_health, check_factor_in_table, explain_factor, generate_doc],
    instructions=SYSTEM_PROMPT_V2.replace("Priya Nair", "Dr Ananya Iyer")
                                 .replace("lead pricing actuary at ABC General",
                                          "appointed actuary at ABC Health")
                                 .replace("rating table", "severity reference table"),
    show_tool_calls=True, markdown=True,
)
health_agent.print_response("Explain the severity drivers for a Surgical claim on a 61+ member.")
# NOTE: check_factor_in_table still reads the MOTOR table — deliberate teaching bug!
# Exercise: generalise the guardrail to take the loader as context, or write
# check_factor_in_table_health. Guardrails must gate the RIGHT source of truth.

## §10 · Five-line port — Life (your turn)
Build `load_mortality_assumptions_life()` — base qx per 1000 with factors for `issue_age_band`, `smoker_status`, `uw_route` — and create Vikram Rao's **Mortality Assumption Documenter**. Fix the guardrail properly this time.

---
**What ships / what stays** — take the shape (prompt, tools, guardrail, assembler); leave the Colab shortcuts (no auth, no retries, hard-coded tables). The hardening list is in the participant playbook.